# Решения: Линейный и бинарный поиск в логах банка

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import math
import statistics
import time
import pandas as pd


def find_csv(name):
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_07_bank_arrays_search/data/" + name


unsorted_df = pd.read_csv(find_csv("bank_transactions_unsorted.csv"))
by_id_df = pd.read_csv(find_csv("bank_transactions_sorted_by_txn_id.csv"))
by_amount_df = pd.read_csv(find_csv("bank_transactions_sorted_by_amount.csv"))
tiny_df = pd.read_csv(find_csv("bank_transactions_tiny.csv"))
COLS = ["txn_id", "amount", "day", "risk_score"]
unsorted_txns = list(unsorted_df[COLS].itertuples(index=False, name=None))
id_txns = list(by_id_df[COLS].itertuples(index=False, name=None))
amount_txns = list(by_amount_df[COLS].itertuples(index=False, name=None))
tiny_txns = list(tiny_df[COLS].itertuples(index=False, name=None))
id_list = [row[0] for row in id_txns]
amount_list = [row[1] for row in amount_txns]
assert id_list == sorted(id_list)
assert amount_list == sorted(amount_list)
print(f"Загружено {len(unsorted_txns)} транзакций; поля кортежа: {COLS}")


## Урок. 1–2. Контракт и linear

In [ ]:
def linear_search(values, target):
    for index, value in enumerate(values):
        if value == target:
            return index
    return -1


def binary_search(values, target):
    left, right = 0, len(values) - 1
    while left <= right:
        mid = (left + right) // 2
        if values[mid] == target:
            return mid
        if values[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1


def lower_bound(values, target):
    left, right = 0, len(values)
    while left < right:
        mid = (left + right) // 2
        if values[mid] < target:
            left = mid + 1
        else:
            right = mid
    return left


def upper_bound(values, target):
    left, right = 0, len(values)
    while left < right:
        mid = (left + right) // 2
        if values[mid] <= target:
            left = mid + 1
        else:
            right = mid
    return left


def selection_sort(values):
    result = list(values)
    for i in range(len(result)):
        smallest = i
        for j in range(i + 1, len(result)):
            if result[j] < result[smallest]:
                smallest = j
        result[i], result[smallest] = result[smallest], result[i]
    return result


def merge_sorted(left, right):
    i = j = 0
    result = []
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    return result + list(left[i:]) + list(right[j:])


def merge_sort(values):
    if len(values) <= 1:
        return list(values)
    mid = len(values) // 2
    return merge_sorted(merge_sort(values[:mid]), merge_sort(values[mid:]))


def median_runtime(function, values, repeats=3):
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        function(list(values))
        samples.append(time.perf_counter() - start)
    return statistics.median(samples)

sample_ids = id_list[:8]
target = sample_ids[5]
expected_index = linear_search(sample_ids, target)
assert expected_index == 5


## Урок. 3. Трасса

In [ ]:
toy = [3, 8, 14, 21, 31, 44, 57]
left, right, trace = 0, len(toy) - 1, []
while left <= right:
    trace.append((left, right)); mid = (left + right) // 2
    if toy[mid] == 31: break
    if toy[mid] < 31: left = mid + 1
    else: right = mid - 1
assert trace[0] == (0, 6)


## Урок. 4. Binary edges

In [ ]:
assert binary_search([], 5) == -1
assert binary_search([5], 5) == 0
assert binary_search(id_list, id_list[-1]) == len(id_list) - 1


## Урок. 5–6. Сравнения

In [ ]:
def linear_steps(values, target):
    for i, value in enumerate(values, 1):
        if value == target: return i - 1, i
    return -1, len(values)

def binary_steps(values, target):
    left, right, steps = 0, len(values) - 1, 0
    while left <= right:
        steps += 1; mid = (left + right) // 2
        if values[mid] == target: return mid, steps
        if values[mid] < target: left = mid + 1
        else: right = mid - 1
    return -1, steps

li, ls = linear_steps(id_list, id_list[700]); bi, bs = binary_steps(id_list, id_list[700])
assert li == bi == 700 and bs < ls


## Урок. 7–9. Инвариант и gate

In [ ]:
SEARCH_INVARIANT = "Бинарный поиск корректен только на отсортированном списке: после сравнения со средним элементом порядок доказывает, в какой половине цель невозможна. Без сортировки отбрасывание половины не обосновано."
probe = unsorted_txns[37]
pos = linear_search([row[0] for row in unsorted_txns], probe[0]); found_row = unsorted_txns[pos]
checks = {"linear_contract": linear_search([], 1) == -1, "binary_edges": binary_search([5], 5) == 0, "binary_fewer_steps": bs < ls, "invariant_written": len(SEARCH_INVARIANT) >= 140}
assert set(checks.values()) == {True}


## ДЗ. Part A

In [ ]:
targets = [unsorted_txns[i][0] for i in (0, 20, 100, 300)] + [-1]
linear_positions = [linear_search([r[0] for r in unsorted_txns], x) for x in targets]
targets_sorted = [id_list[i] for i in (0, 20, 100, 300)] + [-1]
binary_positions = [binary_search(id_list, x) for x in targets_sorted]
edge_checks = [linear_search([], 1) == -1, binary_search([], 1) == -1, linear_search([1], 1) == 0, binary_search([1], 1) == 0, linear_search([1], 2) == -1, binary_search([1], 2) == -1]
assert binary_positions == [0, 20, 100, 300, -1] and all(edge_checks)


## ДЗ. Challenge

In [ ]:
def binary_search_first(values, target):
    index = lower_bound(values, target)
    return index if index < len(values) and values[index] == target else -1

SEARCH_NOTE = "Линейный поиск работает без preprocessing и стоит O(n) на запрос. Сортировка требует предварительной работы, зато бинарный запрос стоит O(log n). Ограничение: порядок должен поддерживаться после обновлений, иначе индекс становится некорректным."
assert binary_search_first([1, 2, 2, 2, 5], 2) == 1 and len(SEARCH_NOTE) >= 220
